# fast_p Distribution Across Training Runs

For each generated Triton kernel we compute

$$\text{fast\_p} = \frac{\text{baseline runtime}}{\text{kernel runtime}}$$

where the *baseline* is the per-problem PyTorch-eager mean runtime from
`timing/A100/baseline_time_torch.json` (KernelBench `fast_p` convention).
`fast_p > 1` ⇒ the kernel is **faster** than the PyTorch baseline.

Only **compiled + correct** kernels with a valid runtime and a known baseline get a
`fast_p`. The notebook then reports **how many kernels fall into each fast_p range**,
per run and overall.

> Set the paths in the **Config** cell, then *Run All*. The fast_p math mirrors
> `reranker/src/data/build_dataset.py` so numbers stay consistent with the dataset builder.


## ⚙️ Config
*The only cell you need to edit.*

In [ ]:
from pathlib import Path

# ── Run folders to analyze (each must contain eval_results.json + *_kernel.py) ──
RUN_DIRS = [
    "/home/jovyan/jan/GuidedResearch/runs/gpt-oss-120b_kernelbook_level5_triton",
    "/home/jovyan/jan/GuidedResearch/runs/gpt-oss-120b_kernelbook_level6_think_triton",
    "/home/jovyan/jan/GuidedResearch/runs/Qwen3-Coder-30B-A3B-Instruct_kernelbook_level6_think_triton",
    "/home/jovyan/jan/GuidedResearch/runs/Qwen3-Coder-30B-A3B-Instruct_kernelbook_level6_triton",
    "/home/jovyan/jan/GuidedResearch/runs/Qwen3-Coder-Next_kernelbook_level5_triton",
]
RUN_DIRS = [Path(p) for p in RUN_DIRS]

# ── Per-problem PyTorch baseline runtimes (A100) ──────────────────────────────
BASELINE_JSON = Path("/home/jovyan/jan/GuidedResearch/timing/A100/baseline_time_torch.json")

# ── fast_p range buckets (upper-exclusive edges; -inf..+inf auto-added) ────────
#    e.g. an edge of 1.0 splits "slower than baseline" from "faster than baseline".
FASTP_EDGES = [0.5, 0.8, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0]

# fast_p @ threshold report (fraction of kernels at least this fast)
FASTP_THRESHOLDS = [1.0, 1.5, 2.0, 3.0, 5.0, 10.0]


## Setup & fast_p Computation
Mirrors the speedup logic in `reranker/src/data/build_dataset.py`.

In [ ]:
import re, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams["figure.dpi"] = 110

# dataviz palette — fixed categorical order (blue, aqua, yellow, green, violet, ...)
CAT = ["#2a78d6", "#1baf7a", "#eda100", "#008300", "#4a3aa7", "#e34948", "#e87ba4", "#eb6834"]
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e6e3"

_LEVEL_RE  = re.compile(r"level[_-]?(\d+)", re.IGNORECASE)
_KERNEL_RE = "level_{lvl}_problem_{pid}_sample_{sid}_kernel.py"


def load_baseline_times(path: Path) -> dict:
    """{level:int -> {problem_id:int -> mean_ms}} from the KernelBench timing JSON."""
    raw = json.loads(Path(path).read_text())
    out = {}
    for level_key, problems in raw.items():
        m = re.match(r"level(\d+)$", str(level_key))
        if not m or not isinstance(problems, dict):
            continue
        per = {}
        for fname, stats in problems.items():
            pm = re.match(r"(\d+)_", str(fname))
            if pm and isinstance(stats, dict) and stats.get("mean") is not None:
                per[int(pm.group(1))] = float(stats["mean"])
        out[int(m.group(1))] = per
    return out


def level_of(run_dir: Path) -> int:
    m = _LEVEL_RE.search(run_dir.name)
    if not m:
        raise ValueError(f"cannot infer level from run dir name: {run_dir.name}")
    return int(m.group(1))


def load_run(run_dir: Path, baseline_times: dict) -> pd.DataFrame:
    """One row per evaluated sample, with fast_p for correct kernels."""
    run_dir = Path(run_dir)
    eval_path = run_dir / "eval_results.json"
    if not eval_path.is_file():
        print(f"⚠️  no eval_results.json in {run_dir} — skipping")
        return pd.DataFrame()

    level = level_of(run_dir)
    base_lvl = baseline_times.get(level, {})
    eval_results = json.loads(eval_path.read_text())

    rows = []
    for pid_str, samples in eval_results.items():
        pid = int(pid_str)
        baseline = base_lvl.get(pid)
        for s in samples:
            sid = int(s["sample_id"])
            # require the staged kernel file to exist (matches dataset builder)
            kpath = run_dir / _KERNEL_RE.format(lvl=level, pid=pid, sid=sid)
            compiled = bool(s.get("compiled", False))
            correct  = bool(s.get("correctness", False))
            rt = s.get("runtime")
            rt = float(rt) if rt is not None else None

            fast_p = None
            if correct and rt is not None and rt > 0 and baseline is not None:
                fast_p = baseline / rt

            if not correct:
                status = "failed/incorrect"
            elif rt is None or rt <= 0:
                status = "correct_no_runtime"
            elif baseline is None:
                status = "correct_no_baseline"
            else:
                status = "correct_with_fast_p"

            rows.append(dict(
                run=run_dir.name, level=level, problem_id=pid, sample_id=sid,
                kernel_file_exists=kpath.is_file(),
                compiled=compiled, correct=correct,
                runtime=rt, baseline=baseline, fast_p=fast_p, status=status,
            ))
    return pd.DataFrame(rows)


baseline_times = load_baseline_times(BASELINE_JSON)
print("baseline levels:", {k: len(v) for k, v in baseline_times.items()})

df = pd.concat([load_run(rd, baseline_times) for rd in RUN_DIRS], ignore_index=True)
print(f"loaded {len(df):,} samples across {df['run'].nunique()} runs")
df.head()


## Correctness Funnel
How many samples survive to the point of having a `fast_p` at all.

In [ ]:
funnel = (df.assign(n=1)
            .groupby("run")
            .agg(samples=("n", "size"),
                 compiled=("compiled", "sum"),
                 correct=("correct", "sum"),
                 with_fast_p=("fast_p", lambda s: s.notna().sum()))
            .assign(level=df.groupby("run")["level"].first()))
funnel["compile_%"]  = (100 * funnel["compiled"]    / funnel["samples"]).round(1)
funnel["correct_%"]  = (100 * funnel["correct"]     / funnel["samples"]).round(1)
funnel["fast_p_%"]   = (100 * funnel["with_fast_p"] / funnel["samples"]).round(1)
funnel = funnel[["level", "samples", "compiled", "correct", "with_fast_p",
                 "compile_%", "correct_%", "fast_p_%"]].sort_index()
funnel.loc["ALL"] = [
    "", len(df), df["compiled"].sum(), df["correct"].sum(), df["fast_p"].notna().sum(),
    round(100*df["compiled"].sum()/len(df), 1),
    round(100*df["correct"].sum()/len(df), 1),
    round(100*df["fast_p"].notna().sum()/len(df), 1),
]
funnel


## fast_p Range Summary — the main deliverable
Counts of **correct kernels** whose `fast_p` falls into each range, per run.
`< 1×` = slower than the PyTorch baseline; `≥ 1×` = faster.

In [ ]:
edges = [-np.inf] + list(FASTP_EDGES) + [np.inf]

def _lbl(lo, hi):
    if lo == -np.inf: return f"< {hi:g}×"
    if hi ==  np.inf: return f"≥ {lo:g}×"
    return f"{lo:g}–{hi:g}×"

labels = [_lbl(edges[i], edges[i+1]) for i in range(len(edges)-1)]

fp = df[df["fast_p"].notna()].copy()
fp["bucket"] = pd.cut(fp["fast_p"], bins=edges, labels=labels,
                      right=False, ordered=True)

# counts: rows = fast_p bucket, cols = run, plus ALL
summary = (fp.groupby(["bucket", "run"], observed=False).size()
             .unstack("run", fill_value=0))
summary = summary.reindex(labels)                 # keep bucket order
summary["ALL"] = summary.sum(axis=1)
summary.loc["TOTAL"] = summary.sum(axis=0)
summary


In [ ]:
# same table as row-normalized percentages (share of each run's fast_p kernels)
pct = summary.drop(index="TOTAL").copy()
pct = (100 * pct / pct.sum(axis=0)).round(1)
pct


## fast_p @ threshold
**Absolute count** of correct kernels reaching at least a given speedup (cumulative),
followed by the same table as a share (%).

In [ ]:
# ── absolute counts: # correct kernels with fast_p >= threshold ───────────────
rows_n = []
for run, g in fp.groupby("run"):
    rows_n.append({"run": run, "level": g["level"].iloc[0], "n_fast_p": len(g),
                   **{f"≥{t:g}×": int((g["fast_p"] >= t).sum()) for t in FASTP_THRESHOLDS}})
rows_n.append({"run": "ALL", "level": "", "n_fast_p": len(fp),
               **{f"≥{t:g}×": int((fp["fast_p"] >= t).sum()) for t in FASTP_THRESHOLDS}})
thr_n = pd.DataFrame(rows_n).set_index("run")
thr_n


In [ ]:
# ── same table as a share (%) of each run's fast_p kernels ────────────────────
rows = []
for run, g in fp.groupby("run"):
    rows.append({"run": run, "level": g["level"].iloc[0], "n_fast_p": len(g),
                 **{f"≥{t:g}×": round(100*(g["fast_p"] >= t).mean(), 1)
                    for t in FASTP_THRESHOLDS}})
rows.append({"run": "ALL", "level": "", "n_fast_p": len(fp),
             **{f"≥{t:g}×": round(100*(fp["fast_p"] >= t).mean(), 1)
                for t in FASTP_THRESHOLDS}})
thr = pd.DataFrame(rows).set_index("run")
thr


## Charts

In [ ]:
# ── grouped bars: fast_p bucket counts per run ────────────────────────────────
plot_df = summary.drop(index="TOTAL").drop(columns="ALL")
runs = list(plot_df.columns)
x = np.arange(len(labels))
w = 0.8 / max(len(runs), 1)

fig, ax = plt.subplots(figsize=(12, 5))
for i, run in enumerate(runs):
    ax.bar(x + i*w - 0.4 + w/2, plot_df[run].values, w,
           label=run, color=CAT[i % len(CAT)], edgecolor="white", linewidth=0.5)

# mark the 1x boundary (baseline parity)
for j, lab in enumerate(labels):
    if lab.startswith("≥ 1") or lab == "1–1.5×":
        ax.axvline(x[j] - 0.5, color=MUTED, ls="--", lw=1, alpha=0.6)
        break

ax.set_xticks(x); ax.set_xticklabels(labels, rotation=0)
ax.set_xlabel("fast_p range  (baseline / kernel runtime)", color=INK)
ax.set_ylabel("# correct kernels", color=INK)
ax.set_title("fast_p distribution by run", color=INK, fontsize=13, fontweight="bold")
ax.legend(fontsize=8, frameon=False)
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


In [ ]:
# ── overall (all runs pooled) fast_p distribution, sequential blue ────────────
tot = summary.loc[[b for b in labels], "ALL"]
seq = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf",
       "#1c5cab", "#184f95", "#104281", "#0d366b"]
fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(labels, tot.values, color=[seq[i % len(seq)] for i in range(len(labels))],
              edgecolor="white", linewidth=0.6)
for b, v in zip(bars, tot.values):
    ax.text(b.get_x()+b.get_width()/2, v, f"{int(v):,}",
            ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xlabel("fast_p range", color=INK)
ax.set_ylabel("# correct kernels (all runs)", color=INK)
ax.set_title("Pooled fast_p distribution", color=INK, fontsize=13, fontweight="bold")
ax.grid(axis="y", color=GRID); ax.set_axisbelow(True)
for s in ("top", "right"): ax.spines[s].set_visible(False)
plt.tight_layout(); plt.show()


## (Optional) Best-sample-per-problem view
Above counts every generated sample. If instead you care about the *best* kernel
found per problem (pass@k-style), this bins the max `fast_p` per (run, problem).

In [ ]:
best = (fp.groupby(["run", "problem_id"])["fast_p"].max().reset_index())
best["bucket"] = pd.cut(best["fast_p"], bins=edges, labels=labels, right=False, ordered=True)
best_summary = (best.groupby(["bucket", "run"], observed=False).size()
                    .unstack("run", fill_value=0).reindex(labels))
best_summary["ALL"] = best_summary.sum(axis=1)
best_summary.loc["TOTAL (problems solved)"] = best_summary.sum(axis=0)
best_summary
